# LINet3 Two-Phase Training: OmniObject3D Pretrain + SUN RGB-D Fine-tune**Phase 1:** Pretrain on OmniObject3D (~252K RGB-D pairs, 87 categories)**Phase 2:** Fine-tune on SUN RGB-D (19 categories, train+test)Each phase produces full training diagnostics. Combined history visualization at the end.---## Checklist Before Running- [ ] **Enable A100 GPU:** Runtime > Change runtime type > A100- [ ] **Upload datasets to Drive:**  - `MyDrive/datasets/OmniObject3D_pretrain_256.tar.gz`  - `MyDrive/datasets/sunrgbd_19_traintest.tar.gz`- [ ] **Expected Runtime:** ~4-6 hours total (Phase 1 + Phase 2)

## 1. Environment Setup & GPU Verification

In [ ]:
# Check GPU availability and specs
import torch
import subprocess

print("=" * 60)
print("GPU VERIFICATION")
print("=" * 60)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

    gpu_name = torch.cuda.get_device_name(0)
    if 'A100' in gpu_name:
        print("\nA100 GPU detected - PERFECT for training!")
    elif 'V100' in gpu_name:
        print("\nV100 GPU detected - Good for training (slower than A100)")
    else:
        print(f"\nGPU: {gpu_name}")
else:
    print("\nNO GPU DETECTED!")
    raise RuntimeError("GPU is required for training")

print("\n" + "=" * 60)

In [ ]:
# Detailed GPU info
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

print("\nGoogle Drive mounted successfully!")
print(f"\nDrive contents:")
!ls -la /content/drive/MyDrive/ | head -20

## 3. Clone Repository to Local Disk (Fast I/O)**Important:** We clone to `/content/` (local SSD) instead of Drive for 10-20x faster I/O

In [ ]:
import os
from pathlib import Path

PROJECT_NAME = "Multi-Stream-Neural-Networks"
GITHUB_REPO = "https://github.com/clingergab/Multi-Stream-Neural-Networks.git"
LOCAL_REPO_PATH = f"/content/{PROJECT_NAME}"

print("=" * 60)
print("REPOSITORY SETUP")
print("=" * 60)

os.chdir('/content')
print(f"Starting in: {os.getcwd()}")

if Path(LOCAL_REPO_PATH).exists() and Path(f"{LOCAL_REPO_PATH}/.git").exists():
    print(f"\nRepo already exists: {LOCAL_REPO_PATH}")
    print(f"Pulling latest changes...")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    print("Repo updated")
else:
    if Path(LOCAL_REPO_PATH).exists():
        print(f"\nRemoving incomplete repo copy...")
        !rm -rf {LOCAL_REPO_PATH}

    print(f"\nCloning from GitHub...")
    !git clone {GITHUB_REPO} {LOCAL_REPO_PATH}

    if not Path(LOCAL_REPO_PATH).exists():
        raise RuntimeError(f"Failed to clone repository to {LOCAL_REPO_PATH}")

    print("Repo cloned successfully")
    os.chdir(LOCAL_REPO_PATH)

print(f"\nWorking directory: {os.getcwd()}")

## 4. Install Dependencies

In [ ]:
# Install required packages
print("Installing dependencies...")

!pip install -q h5py tqdm matplotlib seaborn kornia

import h5py
import tqdm
import matplotlib
import seaborn
import kornia

print("All dependencies installed!")
print(f"   h5py: {h5py.__version__}")
print(f"   matplotlib: {matplotlib.__version__}")
print(f"   kornia: {kornia.__version__}")

## 5. Copy Datasets to Local DiskCopy both OmniObject3D (pretrain) and SUN RGB-D (fine-tune) datasets to local RAM for fast I/O.

In [ ]:
from pathlib import Path
import os

# === OmniObject3D Pretrain Dataset ===
DRIVE_OMNI_TAR = "/content/drive/MyDrive/datasets/OmniObject3D_pretrain_256.tar.gz"
LOCAL_OMNI_PATH = "/dev/shm/sparse_omni_256"

print("=" * 60)
print("OMNIOBJECT3D PRETRAIN DATASET")
print("=" * 60)

if Path(LOCAL_OMNI_PATH).exists() and Path(f"{LOCAL_OMNI_PATH}/class_names.txt").exists():
    print(f"Already on local disk: {LOCAL_OMNI_PATH}")
    n_pt = len(list(Path(LOCAL_OMNI_PATH).rglob("*_rgb.pt")))
    print(f"   RGB-D pairs: {n_pt}")
elif Path(DRIVE_OMNI_TAR).exists():
    print(f"Found on Drive: {DRIVE_OMNI_TAR}")
    tar_name = Path(DRIVE_OMNI_TAR).name
    local_tar = f"/dev/shm/{tar_name}"
    !rsync -ah --info=progress2 {DRIVE_OMNI_TAR} {local_tar}
    print(f"\nExtracting...")
    !tar -xzf {local_tar} -C /dev/shm/ 2>&1 | grep -v "Ignoring unknown extended header"
    !rm {local_tar}
    n_pt = len(list(Path(LOCAL_OMNI_PATH).rglob("*_rgb.pt")))
    print(f"Extracted. RGB-D pairs: {n_pt}")
else:
    raise FileNotFoundError(f"OmniObject3D dataset not found at {DRIVE_OMNI_TAR}")

print(f"\nOmni dataset ready at: {LOCAL_OMNI_PATH}")

In [ ]:
# === SUN RGB-D Dataset ===
DRIVE_SUN_TAR = "/content/drive/MyDrive/datasets/sunrgbd_19_traintest.tar.gz"
LOCAL_SUN_PATH = "/dev/shm/sunrgbd_19_traintest"

print("=" * 60)
print("SUN RGB-D 19-CATEGORY DATASET")
print("=" * 60)

if Path(LOCAL_SUN_PATH).exists():
    print(f"Already on local disk: {LOCAL_SUN_PATH}")
    for split in ['train', 'test']:
        split_dir = Path(f"{LOCAL_SUN_PATH}/{split}/rgb")
        if split_dir.exists():
            count = len(list(split_dir.glob("*.png")))
            print(f"   {split.capitalize()} samples: {count}")
elif Path(DRIVE_SUN_TAR).exists():
    print(f"Found on Drive: {DRIVE_SUN_TAR}")
    !rsync -ah --info=progress2 {DRIVE_SUN_TAR} /dev/shm/sunrgbd_19_traintest.tar.gz
    print(f"\nExtracting...")
    !tar -xzf /dev/shm/sunrgbd_19_traintest.tar.gz -C /dev/shm/ 2>&1 | grep -v "Ignoring unknown extended header"
    !rm /dev/shm/sunrgbd_19_traintest.tar.gz
    for split in ['train', 'test']:
        split_dir = Path(f"{LOCAL_SUN_PATH}/{split}/rgb")
        if split_dir.exists():
            count = len(list(split_dir.glob("*.png")))
            print(f"   {split.capitalize()} samples: {count}")
else:
    raise FileNotFoundError(f"SUN RGB-D dataset not found at {DRIVE_SUN_TAR}")

print(f"\nSUN dataset ready at: {LOCAL_SUN_PATH}")

## 6. Setup Python Path & Imports

In [ ]:
import sys
import os

# Remove cached modules
modules_to_reload = [k for k in sys.modules.keys() if k.startswith('src.')]
for module in modules_to_reload:
    del sys.modules[module]

project_root = '/content/Multi-Stream-Neural-Networks'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Project structure:")
!ls -la {project_root}/src/models/

# Import LiNet and dataloaders
print("\nImporting LiNet, dataloaders, and visualization tools...")
from src.models.linear_integration.li_net3 import li_resnet18
from src.models.linear_integration.li_net3.conv import LIBatchNorm2d
from src.data_utils.omnipretrain_dataset import get_omnipretrain_dataloaders
from src.data_utils.sunrgbd_dataset import get_sunrgbd_dataloaders
from src.training.augmentation_config import AugmentationConfig

# Import visualization suite
from src.utils.visualization import (
    FeatureMapVisualizer,
    StreamContributionVisualizer,
    StreamGradCAM,
    IntegrationWeightVisualizer,
    find_misclassified,
    compare_samples,
    StreamRedundancyAnalyzer,
    PerClassDominanceAnalyzer,
    ActivationDivergenceAnalyzer,
    IntegrationWeightEvolutionVisualizer,
    reset_bn_stats,
)

print("All imports successful!")

In [ ]:
from src.utils.seed import set_seed

SEED = 42
DETERMINISTIC = False

set_seed(SEED, deterministic=DETERMINISTIC)

print(f"Seed: {SEED}, Deterministic: {DETERMINISTIC}")

## 7. Shared Utilities

In [ ]:
import os
from datetime import datetime
from pathlib import Path

# Create checkpoint directory on Google Drive (persistent storage)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
checkpoint_dir = f"/content/drive/MyDrive/linet_checkpoints/omni_sun_{timestamp}"
Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)

STREAM_LABELS = {0: 'RGB', 1: 'Depth'}

print(f"Checkpoint directory: {checkpoint_dir}")


def merge_histories(h1, h2):
    """Merge two training histories (concatenate lists, recurse dicts)."""
    merged = {}
    all_keys = set(h1.keys()) | set(h2.keys())
    for key in all_keys:
        v1 = h1.get(key)
        v2 = h2.get(key)
        if v1 is None and v2 is None:
            merged[key] = None
        elif v1 is None:
            merged[key] = v2
        elif v2 is None:
            merged[key] = v1
        elif isinstance(v1, dict) and isinstance(v2, dict):
            merged[key] = merge_histories(v1, v2)
        elif isinstance(v1, list) and isinstance(v2, list):
            merged[key] = v1 + v2
        else:
            merged[key] = v2
    return merged


def plot_training_curves(history, title_suffix, save_path, stream_labels):
    """Plot 2x3 training diagnostics grid."""
    import matplotlib.pyplot as plt
    import math

    n_streams = len(stream_labels)
    stream_train_colors = ['skyblue', 'lightcoral', 'gold', 'lightgreen', 'plum']
    stream_val_colors = ['blue', 'red', 'orange', 'green', 'purple']
    lr_colors = ['blue', 'red', 'orange', 'purple', 'brown']

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f'Training Diagnostics — {title_suffix}', fontsize=16, fontweight='bold', y=1.02)

    # [0,0] Loss
    axes[0, 0].plot(history['train_loss'], label='Train Loss', linewidth=2)
    if 'val_loss' in history and history['val_loss']:
        axes[0, 0].plot(history['val_loss'], label='Val Loss', linewidth=2, linestyle='--')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training Loss', fontweight='bold')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # [0,1] Accuracy
    axes[0, 1].plot([a*100 for a in history['train_accuracy']], label='Full Model Train', linewidth=2, color='green')
    if 'val_accuracy' in history and history['val_accuracy']:
        axes[0, 1].plot([a*100 for a in history['val_accuracy']], label='Full Model Val', linewidth=2, color='darkgreen', linestyle='--')
    for i in range(n_streams):
        key = f'stream_{i}_train_acc'
        if key in history and history[key]:
            axes[0, 1].plot([a*100 for a in history[key]],
                label=f'{stream_labels[i]} Train', linewidth=1, alpha=0.6, linestyle='--',
                color=stream_train_colors[i % len(stream_train_colors)])
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy (%)')
    axes[0, 1].set_yticks(range(0, 101, 10))
    axes[0, 1].set_title('Training Accuracy', fontweight='bold')
    axes[0, 1].legend(fontsize=9, loc='lower right')
    axes[0, 1].grid(True, alpha=0.3)

    # [0,2] Learning Rate
    if 'learning_rates' in history and history['learning_rates']:
        sampled = history['learning_rates'][::max(1, len(history['learning_rates'])//100)]
        axes[0, 2].plot(sampled, linewidth=2, color='green', label='Base LR')
    for i in range(n_streams):
        key = f'stream_{i}_lr'
        if key in history and history[key]:
            axes[0, 2].plot(history[key], linewidth=1, alpha=0.7, linestyle='--',
                color=lr_colors[i % len(lr_colors)], label=f'{stream_labels[i]} LR')
    axes[0, 2].set_xlabel('Epoch')
    axes[0, 2].set_ylabel('Learning Rate')
    axes[0, 2].set_title('Learning Rate Schedule', fontweight='bold')
    axes[0, 2].legend(fontsize=9)
    axes[0, 2].grid(True, alpha=0.3)

    # [1,0] Gradient Norms
    if 'gradient_norms' in history and history['gradient_norms']:
        grad_epochs = range(len(history['gradient_norms']))
        for i in range(n_streams):
            norms = [d.get(f'stream_{i}', {}).get('mean', 0) for d in history['gradient_norms']]
            axes[1, 0].plot(grad_epochs, norms, label=f'{stream_labels[i]}',
                color=stream_val_colors[i % len(stream_val_colors)], linewidth=1.5)
        shared_norms = [d.get('shared', {}).get('mean', 0) for d in history['gradient_norms']]
        axes[1, 0].plot(grad_epochs, shared_norms, label='Shared', color='gray', linewidth=1.5, linestyle='--')
        axes[1, 0].set_yscale('log')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Gradient Norm (log)')
        axes[1, 0].legend(fontsize=9)
    else:
        axes[1, 0].text(0.5, 0.5, 'No gradient data', ha='center', va='center', transform=axes[1, 0].transAxes)
    axes[1, 0].set_title('Per-Stream Gradient Norms', fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3)

    # [1,1] Stream Contribution
    contrib_key = 'stream_0_train_acc'
    if contrib_key in history and history[contrib_key]:
        baseline_vals = history['train_accuracy']
        for i in range(n_streams):
            other = (i + 1) % n_streams if n_streams == 2 else i
            other_vals = history[f'stream_{other}_train_acc']
            contrib = []
            epochs_eval = []
            for e, (other_acc, base) in enumerate(zip(other_vals, baseline_vals)):
                if not math.isnan(other_acc):
                    contrib.append((base - other_acc) * 100)
                    epochs_eval.append(e)
            axes[1, 1].plot(epochs_eval, contrib,
                label=f'{stream_labels[i]}', color=stream_val_colors[i % len(stream_val_colors)],
                linewidth=1.5, marker='o', markersize=3)
        axes[1, 1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Contribution (pp)')
        axes[1, 1].legend(fontsize=9)
    else:
        axes[1, 1].text(0.5, 0.5, 'No stream data', ha='center', va='center', transform=axes[1, 1].transAxes)
    axes[1, 1].set_title('Per-Stream Contribution', fontweight='bold')
    axes[1, 1].grid(True, alpha=0.3)

    # [1,2] Gradient Health
    if 'gradient_health' in history and history['gradient_health']:
        axes[1, 2].axis('off')
        health_text = "Gradient Health Summary:\n\n"
        status_counts = {}
        for h in history['gradient_health']:
            status = h.get('status', 'unknown') if isinstance(h, dict) else str(h)
            status_counts[status] = status_counts.get(status, 0) + 1
        for status, count in sorted(status_counts.items(), key=lambda x: -x[1]):
            health_text += f"  {status}: {count} epochs\n"
        axes[1, 2].text(0.1, 0.9, health_text, transform=axes[1, 2].transAxes,
            fontsize=10, verticalalignment='top', fontfamily='monospace')
    else:
        axes[1, 2].text(0.5, 0.5, 'No gradient health data', ha='center', va='center', transform=axes[1, 2].transAxes)
    axes[1, 2].set_title('Gradient Health', fontweight='bold')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")


def plot_integration_weights(history, checkpoint_dir, train_config, stream_labels):
    """Plot integration weight evolution and snapshots."""
    evo_viz = IntegrationWeightEvolutionVisualizer(stream_labels=stream_labels)

    if 'integration_weight_norms' in history:
        evo_viz.plot_norm_evolution(history, save_path=f"{checkpoint_dir}/integration_weight_evolution.png")
        print("Integration weight norm evolution saved.")
    else:
        print("No integration weight norm data found.")

    snapshot_dir = train_config.get('integration_snapshot_path')
    if snapshot_dir and os.path.isdir(snapshot_dir) and os.listdir(snapshot_dir):
        evo_viz.plot_snapshot_heatmaps(snapshot_dir, save_path=f"{checkpoint_dir}/integration_weight_snapshots.png")
        print("Integration weight snapshot heatmaps saved.")
    else:
        print("No integration weight snapshots found.")

---# PHASE 1: OmniObject3D PretrainingPretrain on the large OmniObject3D RGB-D dataset to learn general multi-modal features.

## 8. Phase 1 Configuration

In [ ]:
# ======================== PHASE 1: OMNI CONFIG ========================

OMNI_AUGMENTATION = AugmentationConfig(
    rgb_aug_prob=1.0,
    rgb_aug_mag=1.0,
    depth_aug_prob=1.0,
    depth_aug_mag=1.0,
)

OMNI_MODEL_CONFIG = {
    'architecture': 'resnet18',
    'stream_input_channels': [3, 1],  # RGB=3, Depth=1
    'width_multiplier': 0.75,
    'dropout_p': 0.44,
    'device': 'cuda',
    'use_amp': True,
}

OMNI_OPTIMIZER_CONFIG = {
    'stream_lrs': [1e-4, 1e-4],
    'shared_lr': 5e-5,
    'stream_weight_decays': [1e-4, 1e-4],
    'integration_weight_decay': 1e-4,
}

OMNI_TRAIN_CONFIG = {
    'epochs': 30,
    'grad_clip_norm': 1.0,
    'early_stopping': False,
    'restore_best_weights': True,
    'stream_monitoring': True,
    'modality_dropout': True,
    'modality_dropout_start': 0,
    'modality_dropout_ramp': 10,
    'modality_dropout_rate': 0.1,
    'label_smoothing': 0.1,
    'gradient_monitoring': True,
    'gradient_log_freq': 0,
    'track_integration_weights': True,
    'integration_snapshot_freq': 10,
    'save_path': f"{checkpoint_dir}/phase1_best_model.pt",
    'integration_snapshot_path': f"{checkpoint_dir}/phase1_integration_snapshots",
}

print("Phase 1 (Omni) config defined.")
print(f"  Epochs: {OMNI_TRAIN_CONFIG['epochs']}")

## 9. Load OmniObject3D Dataset

In [ ]:
print("=" * 60)
print("LOADING OMNIOBJECT3D PRETRAIN DATASET")
print("=" * 60)

omni_train_loader, omni_val_loader, omni_num_classes = get_omnipretrain_dataloaders(
    data_root=LOCAL_OMNI_PATH,
    batch_size=64,
    num_workers=8,
    seed=SEED,
    normalize=True,
    balanced_sampling=True,
    **OMNI_AUGMENTATION.to_dict(),
)

print(f"\nOmni classes: {omni_num_classes}")
print(f"Train batches: {len(omni_train_loader)}")
print(f"Val batches: {len(omni_val_loader)}")

# Test loading a batch
rgb_batch, depth_batch, label_batch = next(iter(omni_train_loader))
print(f"\nBatch shapes: RGB={rgb_batch.shape}, Depth={depth_batch.shape}, Labels={label_batch.shape}")

## 10. Create Model

In [ ]:
print("=" * 60)
print("MODEL CREATION")
print("=" * 60)

model = li_resnet18(
    num_classes=omni_num_classes,
    stream_input_channels=OMNI_MODEL_CONFIG['stream_input_channels'],
    width_multiplier=OMNI_MODEL_CONFIG['width_multiplier'],
    dropout_p=OMNI_MODEL_CONFIG['dropout_p'],
    device=OMNI_MODEL_CONFIG['device'],
    use_amp=OMNI_MODEL_CONFIG['use_amp'],
)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nLINet3-ResNet18 created")
print(f"  Total parameters: {total_params:,}")
print(f"  Num classes (Omni): {omni_num_classes}")
print(f"  Streams: {STREAM_LABELS}")

## 11. Compile & Train Phase 1 (OmniObject3D)

In [ ]:
import warnings
from src.training.optimizers import create_stream_optimizer
from src.training.schedulers import setup_scheduler

warnings.filterwarnings('ignore', message='The epoch parameter in `scheduler.step\\(\\)` was not necessary', category=UserWarning)

os.makedirs(OMNI_TRAIN_CONFIG['integration_snapshot_path'], exist_ok=True)

# Create optimizer for Phase 1
omni_optimizer = create_stream_optimizer(
    model,
    optimizer_type='adamw',
    stream_lrs=OMNI_OPTIMIZER_CONFIG['stream_lrs'],
    stream_weight_decays=OMNI_OPTIMIZER_CONFIG['stream_weight_decays'],
    shared_lr=OMNI_OPTIMIZER_CONFIG['shared_lr'],
    integration_weight_decay=OMNI_OPTIMIZER_CONFIG['integration_weight_decay'],
)

# No scheduler for short pretrain trials (constant LR)
# Change to setup_scheduler(...) if you want a cosine schedule

# Compile
model.compile(
    optimizer=omni_optimizer,
    scheduler=None,
    loss='cross_entropy',
    label_smoothing=OMNI_TRAIN_CONFIG['label_smoothing'],
    gpu_augmentation=False,
    **OMNI_AUGMENTATION.to_dict(),
)

print("=" * 60)
print("PHASE 1: PRETRAINING ON OMNIOBJECT3D")
print("=" * 60)

history1 = model.fit(
    train_loader=omni_train_loader,
    val_loader=omni_val_loader,
    epochs=OMNI_TRAIN_CONFIG['epochs'],
    verbose=True,
    save_path=OMNI_TRAIN_CONFIG['save_path'],
    early_stopping=OMNI_TRAIN_CONFIG['early_stopping'],
    restore_best_weights=OMNI_TRAIN_CONFIG['restore_best_weights'],
    grad_clip_norm=OMNI_TRAIN_CONFIG['grad_clip_norm'],
    stream_monitoring=OMNI_TRAIN_CONFIG['stream_monitoring'],
    modality_dropout=OMNI_TRAIN_CONFIG['modality_dropout'],
    modality_dropout_start=OMNI_TRAIN_CONFIG['modality_dropout_start'],
    modality_dropout_ramp=OMNI_TRAIN_CONFIG['modality_dropout_ramp'],
    modality_dropout_rate=OMNI_TRAIN_CONFIG['modality_dropout_rate'],
    gradient_monitoring=OMNI_TRAIN_CONFIG['gradient_monitoring'],
    gradient_log_freq=OMNI_TRAIN_CONFIG['gradient_log_freq'],
    track_integration_weights=OMNI_TRAIN_CONFIG['track_integration_weights'],
    integration_snapshot_path=OMNI_TRAIN_CONFIG['integration_snapshot_path'],
    integration_snapshot_freq=OMNI_TRAIN_CONFIG['integration_snapshot_freq'],
)

print("\n" + "=" * 60)
print("PHASE 1 COMPLETE!")
print(f"  Final train loss: {history1['train_loss'][-1]:.4f}")
print(f"  Final train acc:  {history1['train_accuracy'][-1]*100:.2f}%")
if history1.get('val_accuracy'):
    print(f"  Final val acc:    {history1['val_accuracy'][-1]*100:.2f}%")
print("=" * 60)

## 12. Phase 1 Analysis: Training Curves & Integration Weights

In [ ]:
# Phase 1 training curves
plot_training_curves(
    history1,
    title_suffix="Phase 1 (OmniObject3D Pretrain)",
    save_path=f"{checkpoint_dir}/phase1_training_diagnostics.png",
    stream_labels=STREAM_LABELS,
)

# Phase 1 integration weights
plot_integration_weights(history1, checkpoint_dir, OMNI_TRAIN_CONFIG, STREAM_LABELS)

---# PHASE 2: SUN RGB-D Fine-tuningReplace the classification head and fine-tune on SUN RGB-D 19-category dataset.The backbone weights carry over from Phase 1.

## 13. Phase 2 Configuration

In [ ]:
# ======================== PHASE 2: SUN CONFIG ========================

SUN_AUGMENTATION = AugmentationConfig(
    rgb_aug_prob=1.0,
    rgb_aug_mag=1.29,
    depth_aug_prob=0.98,
    depth_aug_mag=1.0,
)

SUN_OPTIMIZER_CONFIG = {
    'stream_lrs': [8.5e-5, 2.6e-4],
    'shared_lr': 7.0e-5,
    'stream_weight_decays': [4.5e-4, 7.8e-5],
    'integration_weight_decay': 1.7e-4,
}

SUN_SCHEDULER_CONFIG = {
    'scheduler_type': 'cosine',
    't_max': 100,
    's1_eta': 8.1e-7,
    's2_eta': 8.5e-7,
    'eta_min': 4.6e-7,
    'warmup_epochs': 5,
    'warmup_start_factor': 0.2,
}

SUN_TRAIN_CONFIG = {
    'epochs': 120,
    'grad_clip_norm': 0.52,
    'early_stopping': False,
    'restore_best_weights': True,
    'stream_monitoring': True,
    'modality_dropout': True,
    'modality_dropout_start': 0,
    'modality_dropout_ramp': 30,
    'modality_dropout_rate': 0.1,
    'label_smoothing': 0.13,
    'gradient_monitoring': True,
    'gradient_log_freq': 0,
    'track_integration_weights': True,
    'integration_snapshot_freq': 10,
    'save_path': f"{checkpoint_dir}/phase2_best_model.pt",
    'integration_snapshot_path': f"{checkpoint_dir}/phase2_integration_snapshots",
}

SUN_NUM_CLASSES = 19

print("Phase 2 (SUN) config defined.")
print(f"  Epochs: {SUN_TRAIN_CONFIG['epochs']}")

## 14. Load SUN RGB-D Dataset

In [ ]:
# Verify dataset structure
print("=" * 60)
print("DATASET STRUCTURE VERIFICATION")
print("=" * 60)

dataset_root = Path(LOCAL_SUN_PATH)

for split in ['train', 'test']:
    split_dir = dataset_root / split
    if split_dir.exists():
        print(f"  {split}/")
        for modality in ['rgb', 'depth']:
            mod_dir = split_dir / modality
            if mod_dir.exists():
                print(f"    {modality}/ - {len(list(mod_dir.glob('*.png')))} images")

# Read class names
class_names_file = dataset_root / 'class_names.txt'
if class_names_file.exists():
    with open(class_names_file, 'r') as f:
        class_names = [line.strip().split(': ', 1)[-1] if ': ' in line.strip() else line.strip() for line in f if line.strip()]
    print(f"\nClasses ({len(class_names)}):")
    for i, name in enumerate(class_names):
        print(f"  {i}: {name}")

In [ ]:
print("=" * 60)
print("LOADING SUN RGB-D 19-CATEGORY DATASET (TRAIN + TEST)")
print("=" * 60)

sun_train_loader, sun_val_loader, sun_test_loader = get_sunrgbd_dataloaders(
    data_root=LOCAL_SUN_PATH,
    batch_size=64,
    num_workers=8,
    seed=SEED,
    **SUN_AUGMENTATION.to_dict(),
    stratified=True,
    normalize=True,
)

# Use test_loader for evaluation (val_loader is None with traintest split)
test_loader = sun_test_loader

print(f"\nSUN dataset loaded!")
print(f"  Train: {len(sun_train_loader.dataset)} samples ({len(sun_train_loader)} batches)")
print(f"  Test: {len(sun_test_loader.dataset)} samples ({len(sun_test_loader)} batches)")

## 15. Replace Classification Head & RecompileThe backbone weights from Phase 1 are preserved. Only the final classification headis replaced to match SUN RGB-D's 19 classes.

In [ ]:
import torch.nn as nn

print("=" * 60)
print("REPLACING CLASSIFICATION HEAD FOR SUN RGB-D")
print("=" * 60)

# Replace the classifier head (num_classes changed)
old_num_classes = omni_num_classes
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, SUN_NUM_CLASSES).to(model.device)

print(f"  Old head: {in_features} -> {old_num_classes} (Omni)")
print(f"  New head: {in_features} -> {SUN_NUM_CLASSES} (SUN)")
print(f"  Backbone weights: PRESERVED from Phase 1")

os.makedirs(SUN_TRAIN_CONFIG['integration_snapshot_path'], exist_ok=True)

# Create fresh optimizer for Phase 2
sun_optimizer = create_stream_optimizer(
    model,
    optimizer_type='adamw',
    stream_lrs=SUN_OPTIMIZER_CONFIG['stream_lrs'],
    stream_weight_decays=SUN_OPTIMIZER_CONFIG['stream_weight_decays'],
    shared_lr=SUN_OPTIMIZER_CONFIG['shared_lr'],
    integration_weight_decay=SUN_OPTIMIZER_CONFIG['integration_weight_decay'],
)

# Create scheduler for Phase 2
sun_scheduler = setup_scheduler(
    sun_optimizer,
    scheduler_type=SUN_SCHEDULER_CONFIG['scheduler_type'],
    epochs=SUN_SCHEDULER_CONFIG['t_max'],
    train_loader_len=len(sun_train_loader),
    t_max=SUN_SCHEDULER_CONFIG['t_max'],
    eta_min=[SUN_SCHEDULER_CONFIG['s1_eta'], SUN_SCHEDULER_CONFIG['s2_eta'],
             SUN_SCHEDULER_CONFIG['eta_min'], SUN_SCHEDULER_CONFIG['eta_min']],
    warmup_epochs=SUN_SCHEDULER_CONFIG['warmup_epochs'],
    warmup_start_factor=SUN_SCHEDULER_CONFIG['warmup_start_factor'],
)

# Recompile with new optimizer and scheduler
model.compile(
    optimizer=sun_optimizer,
    scheduler=sun_scheduler,
    loss='cross_entropy',
    label_smoothing=SUN_TRAIN_CONFIG['label_smoothing'],
    gpu_augmentation=False,
    **SUN_AUGMENTATION.to_dict(),
)

print("\nModel recompiled for Phase 2!")

## 16. Train Phase 2 (SUN RGB-D Fine-tuning)

In [ ]:
print("=" * 60)
print("PHASE 2: FINE-TUNING ON SUN RGB-D")
print("=" * 60)

history2 = model.fit(
    train_loader=sun_train_loader,
    val_loader=None,  # No validation set (train+test split)
    epochs=SUN_TRAIN_CONFIG['epochs'],
    verbose=True,
    save_path=SUN_TRAIN_CONFIG['save_path'],
    early_stopping=SUN_TRAIN_CONFIG['early_stopping'],
    restore_best_weights=SUN_TRAIN_CONFIG['restore_best_weights'],
    grad_clip_norm=SUN_TRAIN_CONFIG['grad_clip_norm'],
    stream_monitoring=SUN_TRAIN_CONFIG['stream_monitoring'],
    modality_dropout=SUN_TRAIN_CONFIG['modality_dropout'],
    modality_dropout_start=SUN_TRAIN_CONFIG['modality_dropout_start'],
    modality_dropout_ramp=SUN_TRAIN_CONFIG['modality_dropout_ramp'],
    modality_dropout_rate=SUN_TRAIN_CONFIG['modality_dropout_rate'],
    gradient_monitoring=SUN_TRAIN_CONFIG['gradient_monitoring'],
    gradient_log_freq=SUN_TRAIN_CONFIG['gradient_log_freq'],
    track_integration_weights=SUN_TRAIN_CONFIG['track_integration_weights'],
    integration_snapshot_path=SUN_TRAIN_CONFIG['integration_snapshot_path'],
    integration_snapshot_freq=SUN_TRAIN_CONFIG['integration_snapshot_freq'],
)

print("\n" + "=" * 60)
print("PHASE 2 COMPLETE!")
print(f"  Final train loss: {history2['train_loss'][-1]:.4f}")
print(f"  Final train acc:  {history2['train_accuracy'][-1]*100:.2f}%")
print("=" * 60)

## 17. Phase 2 Analysis: Training Curves & Integration Weights

In [ ]:
# Phase 2 training curves
plot_training_curves(
    history2,
    title_suffix="Phase 2 (SUN RGB-D Fine-tune)",
    save_path=f"{checkpoint_dir}/phase2_training_diagnostics.png",
    stream_labels=STREAM_LABELS,
)

# Phase 2 integration weights
plot_integration_weights(history2, checkpoint_dir, SUN_TRAIN_CONFIG, STREAM_LABELS)

---# Combined AnalysisMerge Phase 1 and Phase 2 training histories to see the full training trajectory.

## 18. Combined Training Curves (Phase 1 + Phase 2)

In [ ]:
# Merge histories
history = merge_histories(history1, history2)

print(f"Combined history: {len(history['train_loss'])} total epochs")
print(f"  Phase 1: {len(history1['train_loss'])} epochs (Omni)")
print(f"  Phase 2: {len(history2['train_loss'])} epochs (SUN)")

# Combined training curves
plot_training_curves(
    history,
    title_suffix="Combined (Phase 1: Omni + Phase 2: SUN)",
    save_path=f"{checkpoint_dir}/combined_training_diagnostics.png",
    stream_labels=STREAM_LABELS,
)

# Add phase boundary annotation
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
phase1_epochs = len(history1['train_loss'])

# Loss with phase boundary
ax[0].plot(history['train_loss'], linewidth=2)
ax[0].axvline(x=phase1_epochs, color='red', linestyle='--', alpha=0.7, label=f'Phase 2 starts (epoch {phase1_epochs})')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('Loss')
ax[0].set_title('Combined Training Loss', fontweight='bold')
ax[0].legend()
ax[0].grid(True, alpha=0.3)

# Accuracy with phase boundary
ax[1].plot([a*100 for a in history['train_accuracy']], linewidth=2, color='green')
ax[1].axvline(x=phase1_epochs, color='red', linestyle='--', alpha=0.7, label=f'Phase 2 starts (epoch {phase1_epochs})')
ax[1].set_xlabel('Epoch')
ax[1].set_ylabel('Accuracy (%)')
ax[1].set_title('Combined Training Accuracy', fontweight='bold')
ax[1].legend()
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{checkpoint_dir}/combined_phase_boundary.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {checkpoint_dir}/combined_phase_boundary.png")

## 19. Single-Stream Robustness Evaluation

In [ ]:
print("\n" + "=" * 60)
print("SINGLE-STREAM ROBUSTNESS EVALUATION (TEST SET)")
print("=" * 60)
print("\nTesting model performance with missing streams...\n")

print("[1/3] Evaluating with BOTH streams (normal):")
results_both = model.evaluate(test_loader, stream_monitoring=True)
print(f"      Accuracy: {results_both['accuracy']*100:.2f}%")

print("\n[2/3] Evaluating with RGB ONLY (Depth blanked):")
results_rgb_only = model.evaluate(test_loader, stream_monitoring=True, blanked_streams={1})
print(f"      Accuracy: {results_rgb_only['accuracy']*100:.2f}%")

print("\n[3/3] Evaluating with DEPTH ONLY (RGB blanked):")
results_depth_only = model.evaluate(test_loader, stream_monitoring=True, blanked_streams={0})
print(f"      Accuracy: {results_depth_only['accuracy']*100:.2f}%")

print("\n" + "=" * 60)
print("ROBUSTNESS SUMMARY")
print("=" * 60)
print(f"\n  Both streams:  {results_both['accuracy']*100:.2f}%")
print(f"  RGB only:      {results_rgb_only['accuracy']*100:.2f}% (Depth missing)")
print(f"  Depth only:    {results_depth_only['accuracy']*100:.2f}% (RGB missing)")

rgb_degradation = (results_both['accuracy'] - results_rgb_only['accuracy']) * 100
depth_degradation = (results_both['accuracy'] - results_depth_only['accuracy']) * 100
print(f"\n  Degradation when Depth missing: {rgb_degradation:+.2f}%")
print(f"  Degradation when RGB missing:   {depth_degradation:+.2f}%")

## 20. Test Set Evaluation + Pathway Analysis

In [ ]:
print("=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)

results = model.evaluate(data_loader=test_loader, stream_monitoring=True)

print(f"\nTest Results:")
print(f"  Loss: {results['loss']:.4f}")
print(f"  Overall Accuracy: {results['accuracy']*100:.2f}%")

print(f"\nStream-Specific Performance:")
for i in range(len(OMNI_MODEL_CONFIG['stream_input_channels'])):
    other = (i + 1) % 2
    solo_acc = results[f'stream_{other}_blanked_acc']
    print(f"  Stream{i} ({STREAM_LABELS[i]}) Solo Accuracy: {solo_acc*100:.2f}%")
    print(f"  Stream{i} ({STREAM_LABELS[i]}) Contribution: {results[f'stream_{i}_contribution']*100:+.2f}%")

print(f"\n{'='*60}")
print("PATHWAY ANALYSIS")
print(f"{'='*60}")

pathway_analysis = model.analyze_pathways(data_loader=test_loader)

print(f"\nSamples analyzed: {pathway_analysis['samples_analyzed']}")

print("\nAccuracy:")
print(f"  Full model:      {pathway_analysis['accuracy']['full_model']*100:.2f}%")
for i in range(len(OMNI_MODEL_CONFIG['stream_input_channels'])):
    acc = pathway_analysis['accuracy'][f'stream{i}_only']
    contrib = pathway_analysis['accuracy'][f'stream{i}_contribution']
    print(f"  {STREAM_LABELS[i]} only:       {acc*100:.2f}%  (contribution ratio: {contrib:.3f})")

print("\nFeature Norms (mean +/- std):")
for i in range(len(OMNI_MODEL_CONFIG['stream_input_channels'])):
    mean = pathway_analysis['feature_norms'][f'stream{i}_mean']
    std = pathway_analysis['feature_norms'][f'stream{i}_std']
    print(f"  {STREAM_LABELS[i]}:        {mean:.4f} +/- {std:.4f}")
int_mean = pathway_analysis['feature_norms']['integrated_mean']
int_std = pathway_analysis['feature_norms']['integrated_std']
print(f"  Integrated:  {int_mean:.4f} +/- {int_std:.4f}")

print(f"\n{'='*60}")
print("TRAINING SUMMARY (COMBINED)")
print(f"{'='*60}")
print(f"  Phase 1 epochs:      {len(history1['train_loss'])}")
print(f"  Phase 2 epochs:      {len(history2['train_loss'])}")
print(f"  Total epochs:        {len(history['train_loss'])}")
print(f"  Phase 1 final acc:   {history1['train_accuracy'][-1]*100:.2f}%")
print(f"  Phase 2 final acc:   {history2['train_accuracy'][-1]*100:.2f}%")
print(f"  Test accuracy:       {results['accuracy']*100:.2f}%")

## 21. Save Results & Model

In [ ]:
import json
import torch

print("=" * 60)
print("SAVING RESULTS")
print("=" * 60)

# Save training history as JSON
history_path = f"{checkpoint_dir}/training_history.json"

pa_json = {
    'accuracy': {k: float(v) for k, v in pathway_analysis['accuracy'].items()},
    'loss': {k: float(v) for k, v in pathway_analysis['loss'].items()},
    'feature_norms': {k: float(v) for k, v in pathway_analysis['feature_norms'].items()},
    'samples_analyzed': pathway_analysis['samples_analyzed'],
}

json_history = {
    'phase1_epochs': len(history1['train_loss']),
    'phase2_epochs': len(history2['train_loss']),
    'train_loss': [float(x) for x in history['train_loss']],
    'train_accuracy': [float(x) for x in history['train_accuracy']],
    'learning_rates': [float(x) for x in history.get('learning_rates', [])],
    'omni_model_config': {k: str(v) if not isinstance(v, (int, float, bool, type(None))) else v for k, v in OMNI_MODEL_CONFIG.items()},
    'sun_optimizer_config': SUN_OPTIMIZER_CONFIG,
    'sun_scheduler_config': SUN_SCHEDULER_CONFIG,
    'augmentation_config_phase1': OMNI_AUGMENTATION.to_dict(),
    'augmentation_config_phase2': SUN_AUGMENTATION.to_dict(),
    'test_results': {
        'loss': float(results['loss']),
        'accuracy': float(results['accuracy']),
    },
    'pathway_analysis': pa_json,
}

with open(history_path, 'w') as f:
    json.dump(json_history, f, indent=2)
print(f"Training history saved: {history_path}")

# Save final model
final_model_path = f"{checkpoint_dir}/final_model.pt"
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': model.optimizer.state_dict(),
    'scheduler_state_dict': model.scheduler.state_dict() if model.scheduler else None,
    'config': OMNI_MODEL_CONFIG,
    'history': history,
    'test_accuracy': results['accuracy'],
}, final_model_path)
print(f"Final model saved: {final_model_path}")

print(f"\nAll results saved to: {checkpoint_dir}")
!ls -lh {checkpoint_dir}

## 22. Internal CNN Visualization SuiteEverything below runs on the **trained model** with the **SUN RGB-D test set**. Each cell is independent.

In [ ]:
# --- 22a. Feature Map Visualization ---
fm_viz = FeatureMapVisualizer(model, stream_labels=STREAM_LABELS)

test_iter = iter(test_loader)
sample_batch = next(test_iter)
*stream_batches, labels = sample_batch
stream_inputs = [s[0:1].to(model.device) for s in stream_batches]

print(f"Sample label: {labels[0].item()} ({class_names[labels[0].item()] if 'class_names' in dir() else '?'})")

for layer in ['layer1', 'layer4']:
    print(f"\n{'='*60}\n  {layer.upper()} FEATURE MAPS\n{'='*60}")
    fm_viz.visualize(stream_inputs, layer=layer, top_k=8,
                     save_path=f"{checkpoint_dir}/featuremaps_full_{layer}.png")
    for i, label in STREAM_LABELS.items():
        fm_viz.visualize(stream_inputs, layer=layer, mode='stream', stream_idx=i, top_k=8,
                         save_path=f"{checkpoint_dir}/featuremaps_{label.lower()}_{layer}.png")
        fm_viz.visualize(stream_inputs, layer=layer, mode='ablation', stream_idx=i, top_k=8,
                         save_path=f"{checkpoint_dir}/featuremaps_ablation_{label.lower()}_{layer}.png")

for layer in ['layer1', 'layer4']:
    fm_viz.visualize_batch(test_loader, layer=layer, n=32, top_k=8,
                           save_path=f"{checkpoint_dir}/featuremaps_batch_avg_{layer}.png")

print("\nFeature map visualizations complete!")

In [ ]:
# --- 22b. Stream Contribution Decomposition ---
contrib_viz = StreamContributionVisualizer(model, stream_labels=STREAM_LABELS)

for layer in ['layer1', 'layer2', 'layer3', 'layer4']:
    contrib_viz.visualize(stream_inputs, layer=layer,
                          save_path=f"{checkpoint_dir}/contributions_{layer}.png")

contrib_viz.visualize_batch(test_loader, layer='layer4', n=32,
                            save_path=f"{checkpoint_dir}/contributions_batch_layer4.png")

print("Stream contribution decomposition complete!")

In [ ]:
# --- 22c. Stream-Decomposed Grad-CAM ---
gradcam = StreamGradCAM(model, stream_labels=STREAM_LABELS)

gradcam.visualize(stream_inputs, layer='layer4', mode='integrated',
                  save_path=f"{checkpoint_dir}/gradcam_integrated_layer4.png")

for i, label in STREAM_LABELS.items():
    gradcam.visualize(stream_inputs, layer='layer4', mode='stream', stream_idx=i,
                      save_path=f"{checkpoint_dir}/gradcam_{label.lower()}_layer4.png")

gradcam.visualize(stream_inputs, layer='layer4', mode='decomposed',
                  save_path=f"{checkpoint_dir}/gradcam_decomposed_layer4.png")

for layer in ['layer2', 'layer3', 'layer4']:
    gradcam.visualize(stream_inputs, layer=layer, mode='integrated',
                      save_path=f"{checkpoint_dir}/gradcam_integrated_{layer}.png")

print("Grad-CAM visualizations complete!")

In [ ]:
# --- 22d. Integration Weight Visualization ---
iw_viz = IntegrationWeightVisualizer(model, stream_labels=STREAM_LABELS)

iw_viz.visualize_weights(save_path=f'{checkpoint_dir}/integration_weights.png')
iw_viz.visualize_cross_stream(save_path=f'{checkpoint_dir}/integration_cross_stream.png')

ranks = iw_viz.compute_effective_rank()
for layer, r in ranks.items():
    print(f'  {layer}: {[f"{x:.1f}" for x in r]}')

print('Integration weight visualization complete!')

In [ ]:
# --- 22e. Stream Redundancy Analysis ---
redundancy = StreamRedundancyAnalyzer(model, stream_labels=STREAM_LABELS)

sim_results = redundancy.analyze(test_loader, n=128,
    save_path=f'{checkpoint_dir}/stream_redundancy.png')

for layer_name, sim_matrix in sim_results.items():
    print(f'\n{layer_name}:')
    for i in range(sim_matrix.shape[0]):
        row = '  '.join(f'{sim_matrix[i,j]:.3f}' for j in range(sim_matrix.shape[1]))
        print(f'  {STREAM_LABELS.get(i, f"S{i}")}: {row}')

print('Stream redundancy analysis complete!')

In [ ]:
# --- 22f. Per-Class Stream Dominance ---
class_name_map = {i: name for i, name in enumerate(class_names)} if 'class_names' in dir() else None

dominance = PerClassDominanceAnalyzer(model, stream_labels=STREAM_LABELS)

class_dominance = dominance.analyze(test_loader, layer='layer4', class_names=class_name_map,
    save_path=f'{checkpoint_dir}/per_class_dominance.png')

print('\nPer-class stream contribution ratios:')
for cls_idx, ratios in sorted(class_dominance.items()):
    name = class_name_map[cls_idx] if class_name_map else f'Class {cls_idx}'
    ratio_str = ', '.join(f'{STREAM_LABELS.get(i, f"S{i}")}: {r:.2%}' for i, r in enumerate(ratios))
    print(f'  {name}: {ratio_str}')

In [ ]:
# --- 22g. Misclassification Analysis ---
print('--- Finding Misclassified Samples ---')
misclassified = find_misclassified(model, test_loader, n=10)

print(f'Found {len(misclassified)} misclassified samples:')
for i, mc in enumerate(misclassified[:5]):
    true_name = class_names[mc['true_label']] if 'class_names' in dir() else str(mc['true_label'])
    pred_name = class_names[mc['predicted_label']] if 'class_names' in dir() else str(mc['predicted_label'])
    print(f'  [{i}] True: {true_name}, Predicted: {pred_name}, Confidence: {mc["confidence"]:.2%}')

if misclassified:
    mc_sample = misclassified[0]
    mc_inputs = [s.to(model.device) for s in mc_sample['stream_inputs']]
    gradcam.visualize(mc_inputs, layer='layer4', mode='decomposed',
                      save_path=f'{checkpoint_dir}/gradcam_misclassified_0.png')

if misclassified:
    target_class = misclassified[0]['true_label']
    correct_sample = None
    model.eval()
    with torch.no_grad():
        for batch_data in test_loader:
            *stream_batches, targets = batch_data
            stream_batches_dev = [s.to(model.device) for s in stream_batches]
            targets_dev = targets.to(model.device)
            logits = model(stream_batches_dev)
            preds = logits.argmax(dim=1)
            mask = (targets_dev == target_class) & (preds == target_class)
            if mask.any():
                idx = mask.nonzero(as_tuple=True)[0][0].item()
                correct_sample = {
                    'stream_inputs': [s[idx:idx+1].cpu() for s in stream_batches],
                    'true_label': target_class,
                    'predicted_label': target_class,
                    'confidence': torch.softmax(logits[idx], dim=0)[target_class].item(),
                }
                break

    if correct_sample is not None:
        compare_samples(model, correct_sample=correct_sample, misclassified_sample=misclassified[0],
            layer='layer4', stream_labels=STREAM_LABELS, save_path=f'{checkpoint_dir}/compare_samples.png')

print('Misclassification analysis complete!')

In [ ]:
# --- 22h. Train vs Test Activation Divergence ---
div_analyzer = ActivationDivergenceAnalyzer(model)

divergence = div_analyzer.analyze(sun_train_loader, test_loader, n=128,
    save_path=f'{checkpoint_dir}/activation_divergence.png')

for layer_name, metrics in divergence.items():
    print(f'  {layer_name}: MMD={metrics["mmd"]:.4f}')

print('Activation divergence analysis complete!')

In [ ]:
# --- 22i. BN Stats Reset Experiment ---
import copy

original_test_results = model.evaluate(test_loader)
original_acc = original_test_results['accuracy']
print(f'Original test accuracy: {original_acc*100:.2f}%')

print('\n--- Control: Recompute BN stats on TRAIN set ---')
model_control = copy.deepcopy(model)
reset_bn_stats(model_control, sun_train_loader)
control_results = model_control.evaluate(test_loader)
control_acc = control_results['accuracy']
print(f'After train BN reset: {control_acc*100:.2f}% (delta: {(control_acc-original_acc)*100:+.2f}%)')

print('\n--- Oracle: Recompute BN stats on TEST set ---')
model_oracle = copy.deepcopy(model)
reset_bn_stats(model_oracle, test_loader)
oracle_results = model_oracle.evaluate(test_loader)
oracle_acc = oracle_results['accuracy']
print(f'After test BN reset (oracle): {oracle_acc*100:.2f}% (delta: {(oracle_acc-original_acc)*100:+.2f}%)')

oracle_delta = (oracle_acc - original_acc) * 100
if abs(oracle_delta) > 2:
    print(f'\nBN stats drift accounts for ~{oracle_delta:+.1f}% of the gap.')
else:
    print(f'\nBN stats drift is minimal ({oracle_delta:+.1f}%).')

del model_control, model_oracle

## 23. SummaryAll training diagnostics and visualization analyses are saved to the checkpoint directory on Google Drive.**Saved models:**- `phase1_best_model.pt` - Best model from OmniObject3D pretraining- `phase2_best_model.pt` - Best model from SUN RGB-D fine-tuning- `final_model.pt` - Final model with full state dict, optimizer, scheduler, combined history**Saved training diagnostics:**- `phase1_training_diagnostics.png` - Phase 1 training curves- `phase2_training_diagnostics.png` - Phase 2 training curves- `combined_training_diagnostics.png` - Full combined training curves- `combined_phase_boundary.png` - Loss/accuracy with phase boundary marked**Saved data:**- `training_history.json` - Combined history, configs, test results, pathway analysis- `phase1_integration_snapshots/` - Phase 1 integration weight snapshots- `phase2_integration_snapshots/` - Phase 2 integration weight snapshots**Saved visualizations:**- `integration_weight_evolution.png` / `integration_weight_snapshots.png`- `featuremaps_*.png` - Feature visualizations- `contributions_*.png` - Per-stream contributions- `gradcam_*.png` - Spatial attention maps- `integration_weights.png` / `integration_cross_stream.png`- `stream_redundancy.png` - Stream feature similarity- `per_class_dominance.png` - Per-class stream reliance- `activation_divergence.png` - Train/test distribution shift- `compare_samples.png` - Correct vs misclassified comparison